In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, classification_report,
    ConfusionMatrixDisplay
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42


In [ ]:
data_file = Path("FSSAI_Jaggery_3060_ML_Dataset.csv")

if not data_file.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        data_file = Path(next(iter(uploaded.keys())))
    except Exception as exc:
        raise FileNotFoundError(
            "Put FSSAI_Jaggery_3060_ML_Dataset.csv beside this notebook "
            "or upload it in Google Colab."
        ) from exc

df = pd.read_csv(data_file)

required = {
    "State", "Region", "Jaggery_Type", "Form",
    "Packed", "Ecommerce", "Collection_Channel", "Non_Compliant"
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

if len(df) != 3060:
    warnings.warn(f"Expected 3060 rows; found {len(df)}.")

print("Dataset shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print(df["Non_Compliant"].value_counts(dropna=False))


In [ ]:
TARGET = "Non_Compliant"

FEATURES = [
    "State",
    "Region",
    "Jaggery_Type",
    "Form",
    "Packed",
    "Ecommerce",
    "Collection_Channel"
]

X = df[FEATURES].copy()
y = df[TARGET].astype(int).copy()
groups = df["State"].astype(str).copy()

categorical_features = FEATURES

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ],
    remainder="drop"
)

models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=10,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=400,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

pipelines = {
    name: Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", estimator)
    ])
    for name, estimator in models.items()
}


In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training states:", X_train["State"].nunique())
print("Testing states:", X_test["State"].nunique())
print("State overlap:", len(set(X_train["State"]) & set(X_test["State"])))


In [ ]:
results = []
fitted_models = {}

for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    probabilities = None
    if hasattr(pipeline, "predict_proba"):
        probabilities = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Balanced_Accuracy": balanced_accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "ROC_AUC": (
            roc_auc_score(y_test, probabilities)
            if probabilities is not None and y_test.nunique() == 2
            else np.nan
        )
    })
    fitted_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values(
    by=["F1", "Balanced_Accuracy"],
    ascending=False
).reset_index(drop=True)

display(results_df.round(4))


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]

best_predictions = best_model.predict(X_test)

print("Selected model:", best_model_name)
print(classification_report(
    y_test,
    best_predictions,
    target_names=["Compliant", "Non-compliant"],
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_predictions,
    display_labels=["Compliant", "Non-compliant"],
    cmap="Blues"
)
plt.title(f"State-held-out evaluation: {best_model_name}")
plt.tight_layout()
plt.show()


In [ ]:
importance_result = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=15,
    random_state=RANDOM_STATE,
    scoring="f1"
)

importance_df = pd.DataFrame({
    "Feature": FEATURES,
    "Importance_Mean": importance_result.importances_mean,
    "Importance_STD": importance_result.importances_std
}).sort_values("Importance_Mean", ascending=False)

display(importance_df.round(5))

plt.figure(figsize=(8, 5))
sns.barplot(
    data=importance_df,
    x="Importance_Mean",
    y="Feature"
)
plt.title("Permutation feature importance")
plt.tight_layout()
plt.show()


In [ ]:
final_model = pipelines[best_model_name]
final_model.fit(X, y)

joblib.dump(final_model, "jaggery_quality_final_pipeline.joblib")
results_df.to_csv("model_comparison.csv", index=False)
importance_df.to_csv("feature_importance.csv", index=False)

print("Saved model: jaggery_quality_final_pipeline.joblib")
print("Saved metrics: model_comparison.csv")
print("Saved feature importance: feature_importance.csv")


In [ ]:
example_sample = pd.DataFrame([{
    "State": "Odisha",
    "Region": "East",
    "Jaggery_Type": "Cane Jaggery",
    "Form": "Solid",
    "Packed": 1,
    "Ecommerce": 0,
    "Collection_Channel": "Physical-Packed"
}])

predicted_class = int(final_model.predict(example_sample)[0])

if hasattr(final_model, "predict_proba"):
    predicted_probability = float(final_model.predict_proba(example_sample)[0, 1])
else:
    predicted_probability = np.nan

print("Predicted class:", "Non-compliant" if predicted_class == 1 else "Compliant")
print("Predicted probability of non-compliance:", predicted_probability)
